In [1]:
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
class MixedDimensionEmbedding(nn.Module):
    def __init__(self, block_vocab_sizes, base_dim, alpha=0.5):
        super(MixedDimensionEmbedding, self).__init__()
        self.num_blocks = len(block_vocab_sizes)
        self.base_dim = base_dim
        
        self.embeddings = nn.ModuleList()
        self.projections = nn.ModuleList()
        
        v_min = float(min(block_vocab_sizes))
        
        for vocab_size in block_vocab_sizes:
            # Nội suy số chiều động
            raw_dim = base_dim * ((v_min / float(vocab_size)) ** alpha)
            dim = max(1, 2 ** round(math.log2(max(raw_dim, 1e-9))))
            dim = min(dim, base_dim)
            emb_dim = int(dim)
            
            self.embeddings.append(nn.Embedding(vocab_size, emb_dim))
            
            if emb_dim != base_dim:
                self.projections.append(nn.Linear(emb_dim, base_dim, bias=False))
            else:
                self.projections.append(nn.Identity())
                
    def forward(self, cat_x):
        projected_embs = []
        for i in range(self.num_blocks):
            feature_col = cat_x[:, i] 
            emb = self.embeddings[i](feature_col)
            proj = self.projections[i](emb)
            projected_embs.append(proj)
            
        # Trả về khối 3D để dễ dàng tính Dot Product ở DLRM
        # Shape: (batch_size, num_blocks, base_dim)
        return torch.stack(projected_embs, dim=1)

In [3]:
class DLRMMDE(nn.Module):
    def __init__(self, num_dense_features, vocab_sizes, base_dim, bottom_mlp_dims, top_mlp_dims, alpha=0.5):
        super(DLRMMDE, self).__init__()
        self.num_sparse_features = len(vocab_sizes)
        self.base_dim = base_dim
        
        # 1. Thành phần MDE cho Sparse features
        self.mde = MixedDimensionEmbedding(
            block_vocab_sizes=vocab_sizes,
            base_dim=base_dim,
            alpha=alpha
        )
        
        # 2. Bottom MLP xử lý Dense features
        bottom_layers = []
        in_dim = num_dense_features
        for dim in bottom_mlp_dims:
            bottom_layers.append(nn.Linear(in_dim, dim))
            bottom_layers.append(nn.ReLU())
            in_dim = dim
        self.bottom_mlp = nn.Sequential(*bottom_layers)
        
        # Ràng buộc BẮT BUỘC của DLRM: Output dense phải bằng base_dim để nhân vô hướng
        assert bottom_mlp_dims[-1] == base_dim, f"Output của Bottom MLP ({bottom_mlp_dims[-1]}) phải bằng với base_dim ({base_dim})"
        
        # 3. Tính toán số chiều đầu vào cho Top MLP
        # Bao gồm: vector output của dense (base_dim) + số lượng các cặp tương tác
        num_interactions = ((self.num_sparse_features + 1) * self.num_sparse_features) // 2
        interaction_dim = base_dim + num_interactions
        
        # 4. Top MLP
        top_layers = []
        in_dim = interaction_dim
        for dim in top_mlp_dims:
            top_layers.append(nn.Linear(in_dim, dim))
            top_layers.append(nn.ReLU())
            in_dim = dim
        top_layers.append(nn.Linear(in_dim, 1)) # Layer cuối ra Logits
        self.top_mlp = nn.Sequential(*top_layers)

    def forward(self, dense_x, sparse_x):
        # 1. Đi qua Bottom MLP 
        # Đầu ra có shape: (batch_size, base_dim)
        dense_out = self.bottom_mlp(dense_x) 
        
        # 2. Embeddings qua MDE 
        # Đầu ra có shape: (batch_size, num_sparse_features, base_dim)
        cat_embs = self.mde(sparse_x)
        
        # 3. Ghép khối để tính Interactions
        # Cần biến dense_out từ 2D -> 3D để nối với cat_embs
        # (batch_size, base_dim) -> (batch_size, 1, base_dim)
        dense_out_expanded = dense_out.unsqueeze(1) 
        
        # Nối lại: (batch_size, num_sparse_features + 1, base_dim)
        stacked_emb = torch.cat([dense_out_expanded, cat_embs], dim=1)
        
        # 4. Tính Interactions (Dot products) bằng Batch Matrix Multiplication
        interaction_matrix = torch.bmm(stacked_emb, stacked_emb.transpose(1, 2))
        
        # Lấy phần tam giác trên của ma trận tương tác
        num_features = stacked_emb.size(1)
        row_idx, col_idx = torch.triu_indices(num_features, num_features, offset=1)
        interactions = interaction_matrix[:, row_idx, col_idx] # Shape: (batch_size, num_interactions)
        
        # 5. Nối original dense_out với vector interactions
        concat_x = torch.cat([dense_out, interactions], dim=1)
        
        # 6. Đi qua Top MLP và ép về 1D
        out = self.top_mlp(concat_x)
        return out.squeeze(1)

In [4]:
import math

class StandardCriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        
        self.block_vocab_sizes = []
        self.cat_mappers = []
        
        print("Đang đếm Vocabulary Size cho 26 khối Categorical...")
        self._build_vocab()
        
    def _build_vocab(self):
        df = self.data.to_pandas()
        for col in tqdm(self.cat_cols, desc="Building Vocab"):
            unique_vals = df[col].dropna().unique()
            mapper = {val: idx + 1 for idx, val in enumerate(unique_vals)}
            self.cat_mappers.append(mapper)
            self.block_vocab_sizes.append(len(unique_vals) + 1)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            # Nếu giá trị là None hoặc số âm, gán bằng 0
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                # Công thức chuẩn hóa: log(1 + giá_trị)
                dense_x.append(math.log1p(float(val))) 
        
        # Categorical
        cat_ids = []
        for col_idx, col_name in enumerate(self.cat_cols):
            val = row[col_name]
            mapper = self.cat_mappers[col_idx]
            cat_ids.append(mapper.get(val, 0)) 
            
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            torch.tensor(cat_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32)
        )

In [5]:
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, random_split
import glob
import os

# =========================
# 1. Khai báo cột
# =========================
dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

# =========================
# 2. Load parquet local
# =========================
print("Đang tải dữ liệu parquet local...")

# đường dẫn folder chứa các file parquet
data_dir = "/kaggle/input/datasets/huy291/criteo-cleaned-data/data"

# lấy toàn bộ parquet
parquet_files = sorted(glob.glob(os.path.join(data_dir, "*.parquet")))

print(f"Số file parquet: {len(parquet_files)}")

# load bằng HuggingFace datasets
raw_dataset = load_dataset(
    "parquet",
    data_files=parquet_files,
    split="train"
)

print(raw_dataset)

# =========================
# 3. Dataset wrapper
# =========================
dataset = StandardCriteoDataset(
    raw_dataset,
    dense_cols,
    cat_cols
)

# =========================
# 4. Chia train/val
# =========================
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

print(f"Tổng số mẫu: {len(dataset)}")
print(f"Train: {train_size}")
print(f"Val: {val_size}")

# =========================
# 5. DataLoader
# =========================
train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Hoàn tất tạo DataLoader.")

Đang tải dữ liệu parquet local...
Số file parquet: 50


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/34 [00:00<?, ?it/s]

Dataset({
    features: ['label', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'I10', 'I11', 'I12', 'I13', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26'],
    num_rows: 45840617
})
Đang đếm Vocabulary Size cho 26 khối Categorical...


Building Vocab:   0%|          | 0/26 [00:00<?, ?it/s]

Tổng số mẫu: 45840617
Train: 41256555
Val: 4584062
Hoàn tất tạo DataLoader.


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Khởi tạo mô hình DLRM
model = DLRMMDE(
    num_dense_features=len(dense_cols),
    vocab_sizes=dataset.block_vocab_sizes,
    base_dim=64,
    bottom_mlp_dims=[256, 128, 64], # Hãy chắc chắn số cuối cùng bằng base_dim
    top_mlp_dims=[256, 128, 64],
    alpha=0.5
)

# Wrap DataParallel nếu có >= 2 GPU
if num_gpus > 1:
    print(f"Dùng DataParallel trên {num_gpus} GPU")
    model = nn.DataParallel(model)

model = model.to(device)

# Vì đã bỏ sigmoid trong DeepFM, dùng BCEWithLogitsLoss là hoàn toàn chính xác
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [7]:
print("Bắt đầu huấn luyện...")
EPOCHS = 5

for epoch in range(EPOCHS):
    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0
    train_targets = []
    train_preds = []

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")

    for dense_x, cat_x, labels in train_bar:
        # pin_memory + non_blocking giúp transfer CPU→GPU nhanh hơn
        dense_x = dense_x.to(device, non_blocking=True)
        cat_x   = cat_x.to(device,   non_blocking=True)
        labels  = labels.to(device,   non_blocking=True)

        optimizer.zero_grad()
        logits = model(dense_x, cat_x)

        # DataParallel trả về output ghép từ nhiều GPU → shape vẫn đúng
        loss = criterion(logits.squeeze(), labels.float())

        loss.backward()
        optimizer.step()

        # .mean() để lấy scalar khi DataParallel trả về tensor nhiều phần tử
        train_loss += loss.item()

        probs = torch.sigmoid(logits)
        train_targets.extend(labels.detach().cpu().numpy())
        train_preds.extend(probs.detach().cpu().numpy())

        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_auc = roc_auc_score(train_targets, train_preds)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0
    val_targets = []
    val_preds = []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            cat_x   = cat_x.to(device,   non_blocking=True)
            labels  = labels.to(device,   non_blocking=True)

            logits = model(dense_x, cat_x)
            loss = criterion(logits.squeeze(), labels.float())
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            val_targets.extend(labels.cpu().numpy())
            val_preds.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_auc = roc_auc_score(val_targets, val_preds)

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss: {avg_train_loss:.4f} | Train AUC: {train_auc:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\n"
    )

Bắt đầu huấn luyện...


[TRAIN] Epoch 1/5: 100%|██████████| 10073/10073 [23:11<00:00,  7.24it/s, Loss=0.4482]


Epoch 1: Train Loss: 0.4566 | Train AUC: 0.7931 || Val Loss: 0.4505 | Val AUC: 0.8003



[TRAIN] Epoch 2/5: 100%|██████████| 10073/10073 [23:22<00:00,  7.18it/s, Loss=0.4492]


Epoch 2: Train Loss: 0.4372 | Train AUC: 0.8150 || Val Loss: 0.4576 | Val AUC: 0.7932



...

Epoch 3: Train Loss: 0.3992 | Train AUC: 0.8487 || Val Loss: 0.4957 | Val AUC: 0.7687



[TRAIN] Epoch 4/5: 100%|██████████| 10073/10073 [23:24<00:00,  7.17it/s, Loss=0.4236]


Epoch 4: Train Loss: 0.4042 | Train AUC: 0.8438 || Val Loss: 0.5001 | Val AUC: 0.7653



[TRAIN] Epoch 5/5: 100%|██████████| 10073/10073 [23:25<00:00,  7.16it/s, Loss=0.4117]


Epoch 5: Train Loss: 0.4007 | Train AUC: 0.8463 || Val Loss: 0.5395 | Val AUC: 0.7651

